# Session 14: Multi-Modal RAG with Vision-Language Models
### A hands-on, run-from-scratch notebook for student programmers

You already know *text* RAG (Retrieval-Augmented Generation): split documents into chunks, embed them,
store the vectors, retrieve the closest ones for a question, and let an LLM answer. This notebook
extends that idea to **images + text together**, using **Vision-Language Models (VLMs)** — models that
can *see* pictures as well as read text.

**By the end you will have built, end to end:**
1. A **VLM that parses charts/figures into searchable chunks** (ingestion).
2. **Three different retrieval strategies** for mixing images and text, compared side by side.
3. A single **Qdrant**-backed pipeline where a *text* question can retrieve a *chart image*.
4. **Answer generation** that hands the real image pixels to the VLM so it can read exact numbers.
5. A **video + transcript** extension that answers with timestamp citations.

**How this session is organized** — two breakout rooms of roughly 25 minutes each:
- 🤝 **Breakout Room #1 (Sections 0–5):** setup → the demo corpus → the VLM as parser → three retrieval strategies, compared.
- 🤝 **Breakout Room #2 (Sections 6–10):** the unified retriever → generation from pixels → evaluation → video RAG → recap.

Each breakout room ends with ❓ **questions** and a 🏗️ **activity**. Everything you need to answer the
questions is written in this notebook's explanations — read as you run.

**How to run this (read me first):**
1. This repo ships a tiny synthetic dataset in `./data`, so there's nothing to download by hand.
2. Dependencies are managed with **uv**: run `uv sync` in this folder once, then open this notebook with
   the project's environment (`uv run jupyter lab`, or pick this folder's `.venv` as the kernel in Cursor/VS Code).
3. Create a file named **`.env`** next to this notebook (copy `.env.example`) and put your key in it:
   `OPENAI_API_KEY=sk-...`
4. Run the cells **top to bottom**. The first CLIP call downloads a model (~600 MB) once. After that it's fast.

> 💡 **Model-agnostic.** Everything goes through LangChain's `init_chat_model`, so switching from OpenAI
> to Anthropic, Google, or a local model is a one-line change (shown in Section 0).

> ⚠️ **Cost note.** This calls real models. The dataset is tiny (6 images + 6 docs + a short video), so
> a full run is only a handful of API calls, but you *are* spending tokens.


## Contents

**🤝 Breakout Room #1 — Parsing & Cross-Modal Retrieval**
- [0. Setup: environment, secrets, and the model](#sec0)
- [1. From text RAG to multi-modal RAG](#sec1)
- [2. Building this with Cursor (optional developer workflow)](#sec2)
- [3. The demo corpus (ACME Robotics)](#sec3)
- [4. Stage 1 — The VLM as parser & chunker](#sec4)
- [5. Stage 2 — Three retrieval strategies, compared](#sec5)
- ❓ Questions #1–2 · 🏗️ Activity #1

**🤝 Breakout Room #2 — Generation, Evaluation & Video**
- [6. The unified retriever: image + text in one pipeline](#sec6)
- [7. Generation: answering with text *and* images](#sec7)
- [8. Evaluation & gotchas](#sec8)
- [9. In-depth: Video + transcript RAG](#sec9)
- ❓ Questions #3–4 · 🏗️ Activity #2
- [10. Recap: what we built vs. production](#sec10)


---
# 🤝 Breakout Room #1: Parsing & Cross-Modal Retrieval (~25 min)

**Goal:** get the pipeline set up, turn charts into searchable chunks with a VLM, and compare three
ways to retrieve across modalities.

When you finish Section 5, answer ❓ **Questions #1–2** and complete 🏗️ **Activity #1** before moving on.


<a id="sec0"></a>
## 0. Setup: packages, secrets, and the model

Three small steps: install dependencies, load your API key from `.env`, and create the model object.


**Cell 0.1 — Verify your environment (dependencies come from `uv sync`).**
This cohort manages dependencies with **uv**: `pyproject.toml` lists every package this notebook needs,
and `uv sync` installs them into `.venv`. If you opened this notebook with that environment, the check
below passes. If anything is missing, run `uv sync` in this folder and restart the kernel — there is no
`pip install` step anywhere in this notebook.


In [1]:
# Dependencies are installed by `uv sync` (see pyproject.toml) — this cell only verifies them.
import importlib.util

REQUIRED = ["langchain", "langchain_openai", "langchain_qdrant", "qdrant_client",
            "sentence_transformers", "PIL", "matplotlib", "imageio", "cv2", "dotenv"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
assert not missing, f"Missing {missing} — run `uv sync` in this folder, then restart the kernel."
print("✅ Environment OK — all packages available.")


✅ Environment OK — all packages available.


**Cell 0.2 — Load your secret key from `.env` (never hard-code secrets).**
Hard-coding an API key in a notebook is dangerous: it gets committed to git and leaks. Instead we keep
keys in a local `.env` file (which `.gitignore` excludes) and load them with `python-dotenv`. Create the
file by copying `.env.example` to `.env` and filling in your key. We also fail *early* with a clear
message if the key is missing, so you don't hit a confusing error 10 cells later.


In [2]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())   # finds the nearest .env (this folder or a parent) and loads it

# These two strings are the ONLY thing you change to use a different provider or models.
VLM_MODEL   = os.environ.get("VLM_MODEL",   "openai:gpt-4o")            # a vision-capable chat model
EMBED_MODEL = os.environ.get("EMBED_MODEL", "text-embedding-3-small")  # a TEXT embedding model

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY not found.\n"
        "1) Copy .env.example to .env  2) put your key in it:  OPENAI_API_KEY=sk-...\n"
        "3) re-run this cell. (Or export the variable in your shell before launching Jupyter.)")

print(f"✅ Config loaded.  VLM_MODEL={VLM_MODEL}   EMBED_MODEL={EMBED_MODEL}")


✅ Config loaded.  VLM_MODEL=openai:gpt-4o   EMBED_MODEL=text-embedding-3-small


**Cell 0.3 — Create the vision-language model, provider-agnostically.**
`init_chat_model("openai:gpt-4o")` returns a chat model that speaks one standard message format no
matter the vendor. To use a different provider, change `VLM_MODEL` above to e.g.
`"anthropic:claude-3-7-sonnet-latest"` or `"google_genai:gemini-1.5-pro"`, install that provider's
package, and set its key in `.env`. `temperature=0` makes outputs repeatable, which is nice for a lesson.

> **Key idea used throughout:** LangChain's **embedding** interface is *text-only*, but **chat models**
> are multi-modal. That one fact is why Section 5 needs *three* different strategies to retrieve images.


In [3]:
from langchain.chat_models import init_chat_model

def get_vlm():
    return init_chat_model(VLM_MODEL, temperature=0)

vlm = get_vlm()   # build once, reuse everywhere below
print("✅ Vision-language model ready:", VLM_MODEL)


✅ Vision-language model ready: openai:gpt-4o


<a id="sec1"></a>
## 1. From text RAG to multi-modal RAG
Classic text RAG: `documents → chunk → embed → store → retrieve → LLM answer`. Multi-modal RAG changes
three of those boxes and adds a new failure mode:

| Stage | Text RAG | Multi-modal RAG |
|---|---|---|
| **Parse / chunk** | split text | a **VLM reads pages, figures, tables** and emits chunks |
| **Embed** | one text embedder | images need a different representation (a caption, a CLIP vector, or both) |
| **Retrieve** | cosine over text vectors | must retrieve *across modalities* — a text question should pull back a **chart image** |
| **Generate** | stuff text in | stuff **text *and* the actual pixels** into a VLM so it can *read* what it cites |

### The VLM does two different jobs — keep them separate in your head
1. **Parsing / chunking (ingestion, done once):** convert visual content into something searchable
   (a description, an extracted table, OCR text).
2. **Understanding / reading (at query time):** *look at* the retrieved image to answer the question,
   grounded in the actual pixels.

The rest of the notebook is organized around those two jobs.


<a id="sec2"></a>
## 2. Building this with Cursor (optional developer workflow)
If you use **Cursor** (an AI-native IDE), a few habits keep the fast-moving APIs honest:

- **Pin your intent in `.cursor/rules/`** so the AI stops reverting to outdated call signatures, e.g.:
  ```md
  - Use LangChain v1 image content blocks: {"type":"image","source_type":"base64","data":...}
  - Embeddings are TEXT-ONLY; for image vectors use CLIP via sentence-transformers.
  - Vector store: Qdrant (QdrantClient(":memory:") for dev). Every model call goes through init_chat_model(VLM_MODEL).
  - Never hard-code API keys; read from os.environ / .env.
  ```
- **Pull real docs into context** with `@Docs`/`@Web` so refactors target the *current* API.
- **Graduate cells into modules** (`ingest.py`, `retrieve.py`, `generate.py`) once they work.

None of this is required to run the notebook — it's just how you'd turn a lesson into a real project.


<a id="sec3"></a>
## 3. The demo corpus (ACME Robotics)
Real multi-modal RAG runs on PDFs, slides, and screenshots. To keep this repo small and self-contained,
we use a **fully synthetic, fictional company** ("ACME Robotics", CC0 public-domain) that ships in
`./data`. It has six charts and six short documents. **Important design choice:** the exact numbers
(e.g. "$27M revenue") live *only in the chart pixels* — they are **not** written in any text file — so
later we can prove that a text question retrieves an image *and* the model reads the number off it.


**Cell 3.1 — Make sure the dataset exists (create it if missing).**
If you cloned the repo, `./data` already contains the charts and docs, and this cell just confirms them.
If the folder is missing, the same code regenerates it with `matplotlib`, so the notebook is never stuck
waiting on an external download. `os.path.exists(...)` guards mean re-running never overwrites your files.


In [4]:
import os
import matplotlib; matplotlib.use("Agg")   # "Agg" = render to files, no GUI window needed
import matplotlib.pyplot as plt

IMG, TXT = "data/images", "data/text"
os.makedirs(IMG, exist_ok=True); os.makedirs(TXT, exist_ok=True)

# --- three small chart-drawing helpers (only used if the images aren't already on disk) -------
def _bar(path, title, labels, values, ylabel, fmt="{:g}"):
    fig, ax = plt.subplots(figsize=(5.4, 3.4)); bars = ax.bar(labels, values)
    ax.set_title(title, fontsize=12, fontweight="bold"); ax.set_ylabel(ylabel)
    ax.spines[["top","right"]].set_visible(False)
    for b, v in zip(bars, values):
        ax.text(b.get_x()+b.get_width()/2, v, fmt.format(v), ha="center", va="bottom", fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

def _line(path, title, labels, values, ylabel):
    fig, ax = plt.subplots(figsize=(5.4, 3.4)); ax.plot(labels, values, marker="o", linewidth=2)
    ax.set_title(title, fontsize=12, fontweight="bold"); ax.set_ylabel(ylabel)
    ax.spines[["top","right"]].set_visible(False)
    for x, v in zip(labels, values): ax.text(x, v, str(v), ha="center", va="bottom", fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

def _pie(path, title, labels, values):
    fig, ax = plt.subplots(figsize=(5.4, 3.4))
    ax.pie(values, labels=[f"{l} {v}%" for l, v in zip(labels, values)], startangle=90, counterclock=False)
    ax.set_title(title, fontsize=12, fontweight="bold"); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

_charts = {
 "fig_revenue_quarterly.png": lambda p: _bar(p, "ACME Robotics — Quarterly Revenue 2024 ($M)", ["Q1","Q2","Q3","Q4"], [12,18,15,27], "Revenue ($M)"),
 "fig_churn_monthly.png":     lambda p: _bar(p, "Monthly Churn Rate (%)", ["Jan","Feb","Mar","Apr"], [5.1,4.8,6.2,3.9], "Churn (%)"),
 "fig_latency_region.png":    lambda p: _bar(p, "P95 API Latency by Region (ms)", ["US","EU","APAC"], [120,180,240], "Latency (ms)"),
 "fig_headcount_dept.png":    lambda p: _bar(p, "Headcount by Department (FY2024)", ["Eng","Sales","Support","G&A"], [45,30,18,12], "People"),
 "fig_nps_quarterly.png":     lambda p: _line(p, "Net Promoter Score by Quarter", ["Q1","Q2","Q3","Q4"], [22,31,28,41], "NPS"),
 "fig_cloud_cost.png":        lambda p: _pie(p, "Cloud Cost Breakdown (FY2024)", ["Compute","Storage","Network","Other"], [55,20,15,10]),
}
for name, fn in _charts.items():
    p = f"{IMG}/{name}"
    if not os.path.exists(p): fn(p)

_docs = {
 "doc_gtm_strategy.md": "# FY2024 Go-To-Market Strategy\n\nACME Robotics prioritized enterprise accounts in the EU. We built a dedicated solutions-engineering team and a channel partner program. Retention focused on onboarding quality and quarterly business reviews. The largest win was a public-sector contract closed in Q4, the main driver of record fourth-quarter revenue.\n",
 "doc_infrastructure.md": "# Platform Infrastructure & Reliability\n\nThe platform runs a multi-region architecture (US, EU, APAC). We track P95 latency per region as a core SLO and page the on-call engineer when a region exceeds target at peak. In 2024 APAC was persistently the slowest region and generated the most alerts; a regional read cache brought P95 back under target.\n",
 "doc_board_notes.md": "# Board Meeting Notes — Q4 2024\n\n- Revenue was strongest in the final quarter, driven by a large public-sector deal.\n- Churn improved after the onboarding revamp but spiked in March due to a billing bug that double-charged a customer cohort.\n- The board requested a reliability review after the APAC latency alerts.\n",
 "doc_people_ops.md": "# People Operations — FY2024\n\nThe company grew to 105 employees. Hiring was engineering-heavy (45 in Engineering), then Sales (30), Support (18), and G&A (12). A structured onboarding program launched in Q2 is credited with improving retention.\n",
 "doc_product_roadmap.md": "# Product Roadmap & Customer Sentiment\n\nNPS trended up from 22 in Q1 to 41 in Q4, with a dip in Q3 (to 28) attributable to a partial APAC outage. Next year focuses on self-serve onboarding and regional performance improvements.\n",
 "doc_finance_costs.md": "# Cloud Cost Structure — FY2024\n\nCloud spend is compute-dominated: Compute 55%, Storage 20%, Network 15%, Other 10%. A review recommended right-sizing compute in low-traffic regions and committed-use discounts. Network cost should fall once the APAC cache cuts cross-region traffic.\n",
}
for name, body in _docs.items():
    p = f"{TXT}/{name}"
    if not os.path.exists(p):
        with open(p, "w") as f: f.write(body)

print("images:", sorted(os.listdir(IMG)))
print("text  :", sorted(os.listdir(TXT)))


Matplotlib is building the font cache; this may take a moment.


images: ['fig_churn_monthly.png', 'fig_cloud_cost.png', 'fig_headcount_dept.png', 'fig_latency_region.png', 'fig_nps_quarterly.png', 'fig_revenue_quarterly.png']
text  : ['doc_board_notes.md', 'doc_finance_costs.md', 'doc_gtm_strategy.md', 'doc_infrastructure.md', 'doc_people_ops.md', 'doc_product_roadmap.md']


**Cell 3.2 — Load every file into a simple list of records.**
We define a tiny `RawRecord` dataclass (an easy way to make a typed struct) and scan the two folders:
Markdown files become `text` records (we keep their body), PNGs become `image` records (we keep the
path — we'll read the pixels later). Everything downstream iterates over this one list.


In [5]:
from dataclasses import dataclass, field
from typing import Optional
import glob

@dataclass
class RawRecord:
    id: str            # filename, used as a stable identifier
    modality: str      # "text" or "image"
    path: str          # where the file lives on disk
    text: Optional[str] = None   # body text (only for text records)

raw_records = []
for fp in sorted(glob.glob(f"{TXT}/*.md") + glob.glob(f"{TXT}/*.txt")):
    with open(fp) as f:
        raw_records.append(RawRecord(os.path.basename(fp), "text", fp, f.read()))
for fp in sorted(glob.glob(f"{IMG}/*.png")):
    raw_records.append(RawRecord(os.path.basename(fp), "image", fp))

for r in raw_records:
    print(f"{r.modality:6s} {r.id}")


text   doc_board_notes.md
text   doc_finance_costs.md
text   doc_gtm_strategy.md
text   doc_infrastructure.md
text   doc_people_ops.md
text   doc_product_roadmap.md
image  fig_churn_monthly.png
image  fig_cloud_cost.png
image  fig_headcount_dept.png
image  fig_latency_region.png
image  fig_nps_quarterly.png
image  fig_revenue_quarterly.png


**Cell 3.3 — Show what each chart contains and the question it answers.**
The dataset ships a `data_dictionary.json` describing each asset. Printing it here documents the demo
and (in Section 8) becomes our answer key for evaluation.


In [6]:
import json
DD_PATH = "data/data_dictionary.json"
if not os.path.exists(DD_PATH):
    dd = {"images": {
        "fig_revenue_quarterly.png": {"demo_question": "Which quarter had the highest revenue and by how much?", "fact": "Q4 highest at $27M"},
        "fig_churn_monthly.png":     {"demo_question": "In which month was churn highest?", "fact": "March 6.2%"},
        "fig_latency_region.png":    {"demo_question": "What is the P95 latency in APAC?", "fact": "APAC 240ms"},
        "fig_headcount_dept.png":    {"demo_question": "Which department has the most people?", "fact": "Engineering 45"},
        "fig_nps_quarterly.png":     {"demo_question": "How did NPS change over the year?", "fact": "22->41, Q3 dip to 28"},
        "fig_cloud_cost.png":        {"demo_question": "How is cloud spend distributed?", "fact": "Compute 55%"},
    }}
    with open(DD_PATH, "w") as f: json.dump(dd, f, indent=2)
data_dictionary = json.load(open(DD_PATH))

print("Each chart and the question it is designed to answer:\n")
for fn, info in data_dictionary["images"].items():
    print(f"• {fn}\n    Q: {info.get('demo_question','')}\n    (fact hidden in the pixels: {info.get('fact','')})\n")


Each chart and the question it is designed to answer:

• fig_revenue_quarterly.png
    Q: Which quarter had the highest revenue and by how much?
    (fact hidden in the pixels: Q4 highest at $27M; Q1 lowest at $12M)

• fig_churn_monthly.png
    Q: In which month was churn highest?
    (fact hidden in the pixels: March spiked to 6.2%; April best at 3.9%)

• fig_latency_region.png
    Q: What is the P95 latency in APAC?
    (fact hidden in the pixels: APAC worst at 240ms; US best at 120ms)

• fig_headcount_dept.png
    Q: Which department has the most people?
    (fact hidden in the pixels: Engineering largest at 45)

• fig_nps_quarterly.png
    Q: How did NPS change over the year?
    (fact hidden in the pixels: Rose to 41 in Q4; dipped to 28 in Q3)

• fig_cloud_cost.png
    Q: How is cloud spend distributed?
    (fact hidden in the pixels: Compute 55%, Storage 20%, Network 15%, Other 10%)



<a id="sec4"></a>
## 4. Stage 1 — The VLM as parser & chunker
First job of the VLM: turn each **image** into a **structured description** (title, takeaway, data
points). That text is what we'll embed and search. This is the "parsing/chunking" step.


**Cell 4.1 — Helpers to send an image to a chat model.**
Models receive images as base64-encoded bytes inside a "content block". `image_to_b64` reads a file and
encodes it; `image_block` wraps it in the standard LangChain v1 shape; `multimodal_message` bundles a
text prompt with one or more images into a single `HumanMessage`. These three helpers are reused every
time we show the model a picture.


In [7]:
import base64, mimetypes
from langchain_core.messages import HumanMessage, SystemMessage

def image_to_b64(path):
    mime = mimetypes.guess_type(path)[0] or "image/png"          # e.g. "image/png"
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode(), mime          # (base64 string, mime type)

def image_block(path):
    b64, mime = image_to_b64(path)
    # This dict is the provider-agnostic way to attach an image to a message.
    return {"type": "image", "source_type": "base64", "data": b64, "mime_type": mime}

def multimodal_message(prompt, image_paths=()):
    content = [{"type": "text", "text": prompt}] + [image_block(p) for p in image_paths]
    return HumanMessage(content=content)


**Cell 4.2 — Ask the VLM to describe each chart as JSON.**
We prompt the model to return strict JSON (kind, title, takeaway, data points) and parse it. This makes
**one API call per image** (six total). We ask for JSON because structured fields are cleaner to embed
and display than free-form prose. In production you'd add error handling / retries around the JSON parse.


In [8]:
import json

PARSE_PROMPT = ("You are parsing a figure for a retrieval system. Return STRICT JSON with keys: "
                "kind, title, takeaway, data_points (a list of short strings). Read the numbers off "
                "the chart. Be concise and factual. Output JSON only — no markdown, no commentary.")

def vlm_parse_image(path):
    """Job #1 (ingestion): turn one image into a structured, searchable record."""
    resp = vlm.invoke([SystemMessage("Output only JSON."), multimodal_message(PARSE_PROMPT, [path])])
    txt = resp.content if isinstance(resp.content, str) else \
          "".join(b.get("text", "") for b in resp.content if isinstance(b, dict))
    txt = txt.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(txt)

# Parse all six charts once. (This is the step that costs a few API calls.)
parsed_images = {r.id: vlm_parse_image(r.path) for r in raw_records if r.modality == "image"}
print(json.dumps(parsed_images["fig_revenue_quarterly.png"], indent=2))


{
  "kind": "bar chart",
  "title": "ACME Robotics \u2014 Quarterly Revenue 2024 ($M)",
  "takeaway": "Q4 had the highest revenue at $27M.",
  "data_points": [
    "Q1: $12M",
    "Q2: $18M",
    "Q3: $15M",
    "Q4: $27M"
  ]
}


**Real-world chunking tips:** rasterize PDFs **page-by-page** so a citation maps to a page number; keep
a `source_path` (and page/region) on every chunk so you can re-load the *original pixels* at answer time;
ask for **Markdown/JSON tables** rather than prose so numbers survive; and never throw the image away —
parsing-to-text is for *finding* the image, but you still show the real pixels to the model when answering.


**Cell 4.3 — Build one unified list of "chunks".**
We merge text docs and image-captions into a single `Chunk` list. Each chunk carries the text we'll embed
(`content`), which modality it is, and `source_path` back to the original file. For an image, `content`
is the caption we just generated; the pixels stay on disk until generation time.


In [9]:
@dataclass
class Chunk:
    id: str; modality: str; content: str; source_path: str; meta: dict = field(default_factory=dict)

chunks = []
for r in raw_records:
    if r.modality == "text":
        chunks.append(Chunk(r.id, "text", r.text, r.path))
    else:
        p = parsed_images[r.id]
        caption = f"{p['title']}. {p['takeaway']} Data: {', '.join(p['data_points'])}."
        chunks.append(Chunk(r.id, "image", caption, r.path, meta={"parsed": p}))

for c in chunks:
    print(f"[{c.modality:5s}] {c.id:26s} -> {c.content[:64]}...")


[text ] doc_board_notes.md         -> # Board Meeting Notes â€” Q4 2024

- **Revenue** was strongest i...
[text ] doc_finance_costs.md       -> # Cloud Cost Structure â€” FY2024

Cloud spend is **compute-domi...
[text ] doc_gtm_strategy.md        -> # FY2024 Go-To-Market Strategy

ACME Robotics prioritized **ente...
[text ] doc_infrastructure.md      -> # Platform Infrastructure & Reliability

The platform runs a **m...
[text ] doc_people_ops.md          -> # People Operations â€” FY2024

The company grew to **105 employ...
[text ] doc_product_roadmap.md     -> # Product Roadmap & Customer Sentiment

Customer sentiment (NPS)...
[image] fig_churn_monthly.png      -> Monthly Churn Rate (%). Churn rate peaked in March at 6.2%. Data...
[image] fig_cloud_cost.png         -> Cloud Cost Breakdown (FY2024). Compute costs dominate the cloud ...
[image] fig_headcount_dept.png     -> Headcount by Department (FY2024). Engineering has the highest he...
[image] fig_latency_region.png     -> P95 API 

<a id="sec5"></a>
## 5. Stage 2 — Three retrieval strategies, compared
Given a **text question**, how do we retrieve relevant items when some are text and some are images?
Three common designs, each built on **Qdrant** — the same vector database we've used all cohort,
here in `:memory:` mode (a real Qdrant engine running inside the notebook process, no server needed):
- **A — Caption → text embeddings (one store):** images become captions, embed everything with a text
  model. Simple; only as good as the caption.
- **B — Unified CLIP embeddings (shared space):** embed text *and raw pixels* into one vector space, so
  a text query matches an image directly.
- **C — Separate stores, fused (RRF):** a text index + an image index, merged by Reciprocal Rank Fusion.
  Most flexible; each modality uses its best embedder.


### Strategy A — caption → text embeddings (single store)
**Cell 5.1 — Embed all chunks with an OpenAI text embedder and put them in one Qdrant collection.**
Because images are already captions, this is just ordinary text RAG. Note the metadata we attach
(`modality`, `source_path`) — it lets us tell an image chunk from a text chunk after retrieval and
re-load the real pixels later. `similarity_search` embeds the query and returns the nearest chunks.


In [10]:
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

def get_text_embedder():
    # A TEXT embedding model. Swap for another provider by changing this line + EMBED_MODEL.
    return OpenAIEmbeddings(model=EMBED_MODEL)

text_embedder = get_text_embedder()

docs_A = [Document(page_content=c.content,
                   metadata={"id": c.id, "modality": c.modality, "source_path": c.source_path})
          for c in chunks]

# ":memory:" runs a real Qdrant engine inside this process — same API as a production cluster,
# zero setup. In production you'd pass url="https://your-qdrant-host:6333" instead.
store_A = QdrantVectorStore.from_documents(docs_A, text_embedder,
                                           location=":memory:", collection_name="strategy_a")

def retrieve_A(query, k=3):
    return store_A.similarity_search(query, k=k)

print("Query: 'Which quarter had the highest revenue?'")
for d in retrieve_A("Which quarter had the highest revenue?"):
    print(f"  [{d.metadata['modality']:5s}] {d.metadata['id']:26s} {d.page_content[:50]}...")


Query: 'Which quarter had the highest revenue?'


  [image] fig_revenue_quarterly.png  ACME Robotics — Quarterly Revenue 2024 ($M). Q4 ha...
  [text ] doc_board_notes.md         # Board Meeting Notes â€” Q4 2024

- **Revenue** w...
  [image] fig_nps_quarterly.png      Net Promoter Score by Quarter. The Net Promoter Sc...


### Strategy B — unified CLIP embeddings (text and pixels in one space)
**Cell 5.2 — A CLIP wrapper that embeds both text and images.**
CLIP maps a caption like *"a bar chart of revenue"* and the actual PNG to nearby vectors. The first time
this runs it **downloads the CLIP model (~600 MB)**. Because LangChain's embedding interface is
*text-only*, we (1) wrap CLIP's **text** side in an `Embeddings` subclass for the query, and (2) embed
the **images** ourselves. We L2-normalize the vectors and create the Qdrant collection with **cosine** distance, so scores are comparable across text and image vectors.


In [11]:
import numpy as np
from langchain_core.embeddings import Embeddings

class ClipEmbedder:
    """Embeds TEXT and IMAGES into one shared vector space using a CLIP model."""
    def __init__(self, model_name="clip-ViT-B-32"):
        from sentence_transformers import SentenceTransformer   # imported here so setup stays fast
        self.model = SentenceTransformer(model_name)            # first call downloads the weights
    def _norm(self, v):
        v = np.asarray(v, dtype="float32")
        return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-9)
    def embed_text(self, texts):
        return self._norm(self.model.encode(list(texts)))
    def embed_images(self, paths):
        from PIL import Image
        return self._norm(self.model.encode([Image.open(p).convert("RGB") for p in paths]))

clip = ClipEmbedder()

# LangChain-compatible wrapper around CLIP's TEXT tower, used to embed the QUERY.
class ClipTextEmbeddings(Embeddings):
    def __init__(self, clip): self.clip = clip
    def embed_documents(self, texts): return self.clip.embed_text(texts).tolist()
    def embed_query(self, text):      return self.clip.embed_text([text])[0].tolist()


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

C:\aie10\lessons\The-AI-Engineering-Certification-v1.0\14_multimodal_rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\guowi\.cache\huggingface\hub\models--sentence-transformers--clip-ViT-B-32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

**Cell 5.3 — Build a unified index: image vectors from pixels, text vectors from CLIP.**
We precompute image vectors from the raw pixels and inject them with a small `qdrant_store_from_vectors`
helper — LangChain's `from_documents` always embeds text itself, so for *precomputed* vectors we upsert
points with the raw Qdrant client and wrap the collection for querying. The query is embedded by CLIP's
text tower, so a plain-English question is compared directly against image pixels — **cross-modal search
with no caption on the image side.** We run the demo query twice: once against the whole unified
collection, and once with a Qdrant **metadata filter** restricted to images — watch how differently the
two behave.


In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, PointStruct,
                                  Filter, FieldCondition, MatchValue)

def qdrant_store_from_vectors(collection_name, texts, vectors, metadatas, query_embedder):
    """Build a Qdrant collection from PRECOMPUTED vectors, wrapped for LangChain querying.

    LangChain's `from_documents` always embeds `page_content` itself — fine for text, but our image
    vectors come from CLIP looking at PIXELS. So we upsert vectors directly with the raw Qdrant
    client, then wrap the collection in QdrantVectorStore. `query_embedder` only embeds QUERIES.
    """
    client = QdrantClient(":memory:")
    client.create_collection(collection_name=collection_name,
                             vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE))
    client.upsert(collection_name=collection_name, points=[
        PointStruct(id=i, vector=[float(x) for x in v],
                    payload={"page_content": t, "metadata": m})
        for i, (t, v, m) in enumerate(zip(texts, vectors, metadatas))])
    return QdrantVectorStore(client=client, collection_name=collection_name, embedding=query_embedder)

image_chunks = [c for c in chunks if c.modality == "image"]
text_chunks  = [c for c in chunks if c.modality == "text"]

img_vecs = clip.embed_images([c.source_path for c in image_chunks])   # vectors from PIXELS
txt_vecs = clip.embed_text([c.content for c in text_chunks])          # vectors from text

texts_B = [c.content for c in image_chunks] + [c.content for c in text_chunks]
vecs_B  = list(img_vecs) + list(txt_vecs)
metas_B = ([{"id": c.id, "modality": "image", "source_path": c.source_path} for c in image_chunks] +
           [{"id": c.id, "modality": "text",  "source_path": c.source_path} for c in text_chunks])

store_B = qdrant_store_from_vectors("strategy_b", texts_B, vecs_B, metas_B, ClipTextEmbeddings(clip))

def retrieve_B(query, k=3, modality=None):
    """Search the unified CLIP collection; optionally restrict to one modality via payload filter."""
    flt = (Filter(must=[FieldCondition(key="metadata.modality", match=MatchValue(value=modality))])
           if modality else None)
    return store_B.similarity_search(query, k=k, filter=flt)

q = "a bar chart comparing API latency across regions"
print(f"Query: {q!r}\n")
print("Unified search (text + images compete in one ranking):")
for d in retrieve_B(q):
    print(f"  [{d.metadata['modality']:5s}] {d.metadata['id']:26s}")
print("\nImages only (pixels matched directly against the text query):")
for d in retrieve_B(q, modality="image"):
    print(f"  [{d.metadata['modality']:5s}] {d.metadata['id']:26s}")


Query: 'a bar chart comparing API latency across regions'

Unified search (text + images compete in one ranking):
  [text ] doc_infrastructure.md     
  [text ] doc_people_ops.md         
  [text ] doc_board_notes.md        

Images only (pixels matched directly against the text query):
  [image] fig_latency_region.png    
  [image] fig_cloud_cost.png        
  [image] fig_churn_monthly.png     


> **What just happened — the modality gap.** In the unified search you'll likely see **text documents
> outrank every image**, even though the query literally describes a chart. This is CLIP's well-known
> **modality gap**: text embeddings cluster in one region of the space and image embeddings in another,
> so a text query usually lands closer to *any* text than to the best-matching image. The images-only
> search tells the real story: among the pixels, the latency chart ranks first — matched by its pixels,
> no caption involved. That *is* the superpower of unified embeddings, but it's also why a single mixed
> collection is fragile in practice. Two standard fixes: **filter/search per modality** (what we just
> did), or keep **separate stores fused by rank** — exactly Strategy C, coming next. And CLIP still
> won't reliably read "240ms" off the chart — which is why we hand the real pixels to the VLM at answer
> time (Section 7).


### Strategy C — separate stores, fused at query time (RRF)
**Cell 5.4 — Keep a text index and an image index; merge results by rank.**
Each modality uses its best embedder (a strong text model for text, CLIP for images). We can't compare
their raw similarity scores (different scales), so we merge by **Reciprocal Rank Fusion**: an item's
score is the sum of `1/(k + rank)` across the lists it appears in. Robust and simple.


In [13]:
store_C_text = QdrantVectorStore.from_documents(
    [Document(page_content=c.content, metadata={"id": c.id, "modality": "text", "source_path": c.source_path})
     for c in text_chunks], text_embedder, location=":memory:", collection_name="strategy_c_text")

img_texts = [c.content for c in image_chunks]
img_metas = [{"id": c.id, "modality": "image", "source_path": c.source_path} for c in image_chunks]
store_C_img = qdrant_store_from_vectors("strategy_c_images", img_texts, img_vecs, img_metas,
                                        ClipTextEmbeddings(clip))

def reciprocal_rank_fusion(result_lists, k=60):
    scores, keep = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc.metadata["modality"], doc.metadata["id"])
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            keep[key] = doc
    return [keep[key] for key in sorted(scores, key=scores.get, reverse=True)]

def retrieve_C(query, k=3):
    text_hits = store_C_text.similarity_search(query, k=k)
    img_hits  = store_C_img.similarity_search(query, k=k)
    return reciprocal_rank_fusion([text_hits, img_hits])[:k]

print("Query: 'why did on-call get paged about APAC?'")
for d in retrieve_C("why did on-call get paged about APAC?"):
    print(f"  [{d.metadata['modality']:5s}] {d.metadata['id']:26s}")


Query: 'why did on-call get paged about APAC?'


  [text ] doc_infrastructure.md     
  [image] fig_latency_region.png    
  [text ] doc_board_notes.md        


### Tradeoffs at a glance
| | **A. Caption → text** | **B. Unified CLIP** | **C. Separate + fuse** |
|---|---|---|---|
| Cross-modal search | via captions only | **native (pixels)** | native (image store) |
| Ingestion cost | **high** (a VLM call per image) | low | medium |
| Fine-grained numbers | as good as the caption | weak (CLIP can't read values) | text strong, image weak |
| Query latency | low | low | higher (2 searches + fuse) |
| Infra simplicity | **simplest** | simple | most moving parts |
| Best when… | text-heavy, few figures | image-dominant corpora | you want each modality's best embedder |
| Main failure mode | caption misses the ask | modality gap (text crowds out images); can't read precise values | rank calibration across stores |

**Practical default:** Strategy **C** for breadth, *always* paired with a VLM at generation so exact
figures are read from pixels. A is a great start; B shines on image-heavy corpora.


---
## 🧗 Breakout Room #1 — Check your understanding

Answer in the cells below. Everything you need is in the sections you just ran — if you're stuck,
re-read the **key idea** note in Section 0.3, the strategy descriptions in Section 5, and the callout
right under Strategy B's results.

### ❓ Question #1:

LangChain's embedding interface is *text-only*, but chat models are multi-modal. Why does that one fact
make image retrieval harder, and what are the **three workarounds** this notebook demonstrates?

#### ✅ Answer:

LangChain's standard `Embeddings` contract accepts strings, so it cannot directly pass image pixels
to an image encoder or store a vector produced from an image. A multi-modal chat model can inspect an
image once it is placed in a message, but that does not by itself make the image searchable in a vector
store. The notebook demonstrates three practical workarounds:

1. **Strategy A — caption then embed:** use the VLM at ingestion to turn each image into searchable text,
   then use an ordinary text embedder for both captions and documents.
2. **Strategy B — precompute unified CLIP vectors:** encode image pixels with CLIP's image tower, upsert
   those vectors directly into Qdrant, and wrap CLIP's text tower in LangChain's `Embeddings` interface
   so a text query lands in the same vector space.
3. **Strategy C — separate modality-specific stores:** use a text embedder for the text collection and
   CLIP for the image collection, search both, and merge their ranked results with RRF.


### ❓ Question #2:

Strategy B's images-only search proves CLIP can find the latency chart *from pixels alone* — yet in the
unified search, text documents usually outrank it (the **modality gap**), and on its own Strategy B
would also do a poor job answering *"What is the P95 latency in APAC?"*. Explain **both** weaknesses,
and name the part of the full pipeline that compensates for each.

#### ✅ Answer:

The first weakness is the **modality gap**. Even though CLIP was trained to align text and images, its
text and image vectors can occupy different regions. In one mixed collection, a text query can therefore
rank semantically weaker text chunks above the correct image. The pipeline compensates by searching with
a modality filter or, preferably here, by using Strategy C's separate text/image searches plus RRF.

The second weakness is that CLIP is a coarse semantic retriever, not a reliable OCR or chart-reading
model. It can recognize a latency chart but cannot be trusted to extract the exact `240 ms` APAC value.
The generation stage compensates by reloading the retrieved chart through `source_path` and sending its
actual pixels to the VLM, which reads the value and produces a grounded, cited answer.


### 🏗️ Activity #1: Probe the three strategies with your own queries

Write **two of your own questions** about the ACME corpus (the data dictionary printout in Section 3 is
your menu — try one *visual* phrasing like "a pie chart of…" and one *factual* phrasing). Run them
through all three strategies with the cell below, then record in the markdown cell after it: **where did
the rankings differ, and why would you expect that given the tradeoffs table?**


In [14]:
# 🏗️ Activity #1 — one visually phrased query and one factual query.
activity_1_cases = {
    "Show me the pie chart that breaks down ACME's cloud spending.": "fig_cloud_cost.png",
    "Which quarter posted the lowest revenue, and what was the amount?": "fig_revenue_quarterly.png",
}
my_queries = list(activity_1_cases)

for q in my_queries:
    print("Q:", q)
    expected = activity_1_cases[q]
    for name, fn in [("A", retrieve_A), ("B", retrieve_B), ("C", retrieve_C)]:
        ids = [d.metadata["id"] for d in fn(q, k=3)]
        print(f"  Strategy {name}: {ids} | target in top 3: {expected in ids}")
    print("-" * 78)


Q: Show me the pie chart that breaks down ACME's cloud spending.


  Strategy A: ['fig_cloud_cost.png', 'doc_finance_costs.md', 'fig_revenue_quarterly.png'] | target in top 3: True
  Strategy B: ['doc_people_ops.md', 'doc_finance_costs.md', 'doc_board_notes.md'] | target in top 3: False


  Strategy C: ['doc_finance_costs.md', 'fig_cloud_cost.png', 'doc_gtm_strategy.md'] | target in top 3: True
------------------------------------------------------------------------------
Q: Which quarter posted the lowest revenue, and what was the amount?


  Strategy A: ['fig_revenue_quarterly.png', 'doc_board_notes.md', 'fig_nps_quarterly.png'] | target in top 3: True
  Strategy B: ['doc_people_ops.md', 'doc_board_notes.md', 'doc_finance_costs.md'] | target in top 3: False


  Strategy C: ['doc_board_notes.md', 'fig_revenue_quarterly.png', 'doc_gtm_strategy.md'] | target in top 3: True
------------------------------------------------------------------------------


**Your observations:**

These probes tested different retrieval signals. The cloud-spend query used visual language (`pie chart`),
while the revenue query asked for an exact fact whose value exists only in the chart pixels.

The observed rankings matched the strategy tradeoffs. **Strategy A retrieved both target figures at rank 1**,
showing that the VLM captions preserved the chart type and the quarterly values. **Strategy B missed both
figures in its unified top three** and returned only text documents, demonstrating the modality gap even for
a visually phrased query. **Strategy C retrieved both target figures at rank 2**, behind a related text document,
because its independent image search gave each figure a route into the RRF result. The VLM must still read the
revenue chart's original pixels to ground the exact `$12M` answer rather than relying on CLIP.

---
#### 🛑 End of Breakout Room #1 — check in with your group, then continue to Breakout Room #2.


---
# 🤝 Breakout Room #2: Generation, Evaluation & Video (~25 min)

**Goal:** wrap the strategies behind one retriever, generate answers grounded in the *real pixels*,
measure retrieval quality, and extend the whole pipeline to video with timestamp citations.

When you finish Section 9, answer ❓ **Questions #3–4** and complete 🏗️ **Activity #2**.


<a id="sec6"></a>
## 6. The unified retriever: image + text in one pipeline
**Cell 6.1 — Wrap a strategy behind one clean `retrieve()` method.**
This small class hides which strategy (A/B/C) is in use and returns plain dicts the generator understands.
Swapping strategies is now a one-word change (`strategy="A" | "B" | "C"`), which is exactly what you want
when experimenting.


In [15]:
class MultiModalRetriever:
    def __init__(self, strategy="C", k=3):
        self.strategy, self.k = strategy, k
        self._fn = {"A": retrieve_A, "B": retrieve_B, "C": retrieve_C}[strategy]
    def retrieve(self, query):
        return [{"id": d.metadata["id"], "modality": d.metadata["modality"],
                 "text": d.page_content, "source_path": d.metadata.get("source_path")}
                for d in self._fn(query, k=self.k)]

retriever = MultiModalRetriever(strategy="C", k=3)
retriever.retrieve("Which quarter had the highest revenue and by how much?")


[{'id': 'doc_board_notes.md',
  'modality': 'text',
  'text': '# Board Meeting Notes â€” Q4 2024\n\n- **Revenue** was strongest in the final quarter, driven by a large public-sector deal.\n- **Churn** improved materially after the onboarding revamp, but **spiked in March** due to a\n  billing bug that double-charged a cohort of customers before it was fixed.\n- The board asked for a deeper reliability review after the APAC latency alerts.\n',
  'source_path': 'data/text\\doc_board_notes.md'},
 {'id': 'fig_revenue_quarterly.png',
  'modality': 'image',
  'text': 'ACME Robotics — Quarterly Revenue 2024 ($M). Q4 had the highest revenue at $27M. Data: Q1: $12M, Q2: $18M, Q3: $15M, Q4: $27M.',
  'source_path': 'data/images\\fig_revenue_quarterly.png'},
 {'id': 'doc_gtm_strategy.md',
  'modality': 'text',
  'text': '# FY2024 Go-To-Market Strategy\n\nACME Robotics prioritized **enterprise accounts in the EU** during 2024. We stood up a\ndedicated solutions-engineering team and launched a chan

<a id="sec7"></a>
## 7. Generation: answering with text *and* images
Second job of the VLM: **read the retrieved images to answer.** We build a prompt containing the
retrieved text chunks (as text) and the retrieved image chunks **as real pixels** (re-loaded from
`source_path`). Now the model can read the exact number off the chart — solving CLIP's "can't read
values" limitation — and we ask it to cite the chunk ids it used.


**Cell 7.1 — Assemble a multi-modal prompt and call the model.**
`build_answer_message` interleaves text snippets and images into one `HumanMessage`. `answer()` retrieves,
builds the message, and invokes the VLM. The system prompt tells the model to stay grounded in the
provided context and to cite ids like `[fig_revenue_quarterly.png]`.


In [16]:
ANSWER_SYSTEM = ("You are a grounded assistant. Answer ONLY from the provided context (text and images). "
                 "Read any figures carefully for exact values. Cite the chunk ids you used like [id]. "
                 "If the answer isn't in the context, say so.")

def build_answer_message(query, retrieved):
    parts = [{"type": "text", "text": f"Question: {query}\n\n--- CONTEXT ---"}]
    for r in retrieved:
        if r["modality"] == "text":
            parts.append({"type": "text", "text": f"\n[{r['id']}] (text): {r['text']}"})
        else:
            parts.append({"type": "text", "text": f"\n[{r['id']}] (image below):"})
            parts.append(image_block(r["source_path"]))     # <-- the ACTUAL pixels go to the model
    parts.append({"type": "text", "text": "\n--- END CONTEXT ---\nAnswer with citations."})
    return HumanMessage(content=parts)

def answer(query, strategy="C", k=3):
    retrieved = MultiModalRetriever(strategy, k).retrieve(query)
    resp = vlm.invoke([SystemMessage(ANSWER_SYSTEM), build_answer_message(query, retrieved)])
    return resp.content if isinstance(resp.content, str) else \
           "".join(b.get("text", "") for b in resp.content if isinstance(b, dict))


**Cell 7.2 — Ask a few real questions.**
The first question's answer lives *only in a chart image*; watch the model read it from the pixels.


In [17]:
for q in [
    "Which quarter had the highest revenue and by how much?",   # answer lives in an IMAGE
    "In which month was churn highest, and what caused it?",     # image + text together
    "How is our cloud spend distributed?",                       # pie-chart IMAGE
    "How did NPS change over the year and when did it dip?",      # image + text
]:
    print("Q:", q)
    print("A:", answer(q, strategy="C"))
    print("-" * 88)


Q: Which quarter had the highest revenue and by how much?


A: The highest revenue was in Q4, with $27 million. This was $9 million more than the next highest quarter, Q2, which had $18 million in revenue [fig_revenue_quarterly.png].
----------------------------------------------------------------------------------------
Q: In which month was churn highest, and what caused it?


A: Churn was highest in March, with a rate of 6.2%. This spike was caused by a billing bug that double-charged a cohort of customers before it was fixed [doc_board_notes.md].
----------------------------------------------------------------------------------------
Q: How is our cloud spend distributed?


A: Our cloud spend is distributed as follows: Compute accounts for 55%, Storage for 20%, Network for 15%, and Other costs for 10% [doc_finance_costs.md].
----------------------------------------------------------------------------------------
Q: How did NPS change over the year and when did it dip?


A: The Net Promoter Score (NPS) increased over the year from 22 in Q1 to 41 in Q4. However, there was a dip in Q3 when the NPS fell to 28, which was attributed to a partial outage in the APAC region [doc_product_roadmap.md].
----------------------------------------------------------------------------------------


Notice the division of labor: **retrieval** finds the revenue chart, and **generation** reads "$27M" off
the pixels. That number is in no text document. This "retrieve to find the chart, look at the chart to
answer" split is the core pattern of production multi-modal RAG.


<a id="sec8"></a>
## 8. Evaluation & gotchas
Measure two things separately: **retrieval** (did the right chunk come back? → recall@k) and **answer**
quality (grounded and correct? → human review or an LLM-as-judge).


**Cell 8.1 — A tiny retrieval eval: recall@3 for each strategy.**
`gold` maps a question to the chunk id(s) that should be retrieved. `recall_at_k` counts a hit if any
gold id appears in the top-k. On this tiny curated corpus you should see high numbers — the point is the
harness and the A/B/C comparison, which becomes more telling on a larger, messier corpus.


In [18]:
gold = {
    "Which quarter had the highest revenue?":       {"fig_revenue_quarterly.png"},
    "In which month was churn highest?":            {"fig_churn_monthly.png"},
    "What is the P95 latency in APAC?":             {"fig_latency_region.png"},
    "Which department has the most people?":        {"fig_headcount_dept.png"},
    "When did NPS dip during the year?":            {"fig_nps_quarterly.png", "doc_product_roadmap.md"},
    "How is cloud spend distributed?":              {"fig_cloud_cost.png"},
    "What was the FY2024 go-to-market focus?":      {"doc_gtm_strategy.md"},
    "What caused the March churn spike?":           {"doc_board_notes.md", "fig_churn_monthly.png"},
}

def recall_at_k(strategy, k=3):
    r = MultiModalRetriever(strategy, k); hits = 0
    for q, want in gold.items():
        got = {d["id"] for d in r.retrieve(q)}
        hits += len(want & got) > 0
    return hits / len(gold)

for s in ["A", "B", "C"]:
    print(f"Strategy {s}: recall@3 = {recall_at_k(s):.2f}")


Strategy A: recall@3 = 1.00


Strategy B: recall@3 = 0.25


Strategy C: recall@3 = 1.00


### Gotchas that bite people
- **Caption bottleneck (A).** If the parser didn't mention it, retrieval can't find it. Enrich the parse
  prompt or add strategy B/C.
- **CLIP can't read (B).** Great at "find the revenue chart", useless for "what was Q3 exactly". Keep the
  pixels available at generation.
- **The modality gap (B).** In one mixed collection, text queries sit closer to text than to
  images. Filter per modality, or fuse separate stores by rank (Strategy C), instead of trusting one
  unified ranking.
- **Score scales don't compare (C).** Use rank-based fusion (RRF), not raw-score thresholds, across
  different embedders.
- **Token blow-up.** Images cost many tokens — cap how many image chunks you send, downscale them, and
  consider prompt caching.
- **Lost provenance.** No `source_path`/page → no pixels and no citation. Carry it from the first parse.
- **Stale index.** When source docs change, regenerate the affected images *and* their captions/vectors
  together, then rebuild the Qdrant collection so the two never drift apart.


<a id="sec9"></a>
## 9. In-depth: extending to video + transcripts
A video is just **two aligned streams**: an **audio track** → a timestamped transcript, and a **sequence
of frames** → images. Video RAG builds **time-windowed chunks** pairing *what was said* with *what was on
screen*, then reuses Sections 5–7. The payoff: answers that cite a **timestamp** ("at 00:26 …").

The dataset ships a ~45s narrated-slide video and its transcript, so this runs with no extra downloads.


**Cell 9.1 — Create the demo video + transcript if they're missing.**
Like the corpus in Section 3, the video ships in `./data`; this cell only rebuilds it (from the charts,
using PIL + imageio) if it isn't already there. In a real project this whole cell is replaced by "point
at your own `.mp4`".


In [19]:
import os, json
import numpy as np
from PIL import Image, ImageDraw, ImageFont

VID, KF = "data/video", "data/video/keyframes"
os.makedirs(KF, exist_ok=True)
MP4 = f"{VID}/fy2024_review.mp4"; TRANSCRIPT = f"{VID}/fy2024_review_transcript.json"

if not (os.path.exists(MP4) and os.path.exists(TRANSCRIPT)):
    W, H, FPS = 1280, 720, 8
    def _font(sz):
        for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                  "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]:
            if os.path.exists(p): return ImageFont.truetype(p, sz)
        return ImageFont.load_default()
    def _slide(title, chart, caption):
        im = Image.new("RGB", (W, H), (255,255,255)); d = ImageDraw.Draw(im)
        d.rectangle([0,0,W,96], fill=(28,42,74)); d.text((40,28), title, font=_font(40), fill=(255,255,255))
        if chart:
            ch = Image.open(chart).convert("RGB"); ch.thumbnail((900,500)); im.paste(ch, ((W-ch.width)//2, 130))
        if caption: d.text((40, H-70), caption, font=_font(24), fill=(40,40,40))
        return im
    timeline = [
     ("ACME Robotics — FY2024 Review", None, "Fiscal Year 2024 Business Review",
      "Welcome to the ACME Robotics fiscal year 2024 review. We'll cover revenue, reliability, and team growth.", None, 6),
     ("Revenue", f"{IMG}/fig_revenue_quarterly.png", "Record Q4 revenue of $27M",
      "Revenue reached a record twenty-seven million dollars in the fourth quarter, driven by a large public-sector contract, up from twelve million in the first quarter.", "fig_revenue_quarterly.png", 10),
     ("Retention", f"{IMG}/fig_churn_monthly.png", "March churn spike from a billing bug",
      "Monthly churn spiked to six point two percent in March because of a billing bug, then improved to three point nine percent in April after the onboarding revamp.", "fig_churn_monthly.png", 10),
     ("Reliability", f"{IMG}/fig_latency_region.png", "APAC P95 latency was the worst region",
      "On reliability, APAC ninety-fifth percentile latency reached two hundred forty milliseconds, the worst of any region, which repeatedly triggered on-call pages. A regional cache brought it back to target.", "fig_latency_region.png", 11),
     ("Team", f"{IMG}/fig_headcount_dept.png", "Grew to 105 people, engineering-heavy",
      "Finally, the team grew to a hundred and five people, with engineering the largest department at forty-five.", "fig_headcount_dept.png", 8),
    ]
    import imageio.v2 as imageio
    frames, segs, t = [], [], 0.0
    for i,(title, chart, cap, text, asset, dur) in enumerate(timeline):
        arr = np.asarray(_slide(title, chart, cap)); frames += [arr]*int(dur*FPS)
        segs.append({"id": f"seg_{i:03d}", "start": round(t,2), "end": round(t+dur,2), "text": text, "onscreen_asset": asset})
        Image.fromarray(arr).save(f"{KF}/frame_{int(t+dur/2):04d}.jpg", quality=85)
        t += dur
    imageio.mimwrite(MP4, frames, fps=FPS, codec="libx264", quality=7, macro_block_size=None)
    json.dump({"video": "fy2024_review.mp4", "fps": FPS, "segments": segs}, open(TRANSCRIPT, "w"), indent=2)

print("video:", os.path.getsize(MP4)//1024, "KB | transcript segments:",
      len(json.load(open(TRANSCRIPT))["segments"]))


video: 113 KB | transcript segments: 5


**Cell 9.2 — Load the timestamped transcript.**
In a real project you'd produce this by running speech-to-text (e.g. Whisper) on the audio; the commented
line shows how. Here we load the JSON the dataset ships. Each segment is `(start, end, text)`.


In [20]:
# Real audio -> transcript would be:  segs = whisper.load_model("base").transcribe("my.mp4")["segments"]
tj = json.load(open(TRANSCRIPT))
transcript_segments = [(s["start"], s["end"], s["text"]) for s in tj["segments"]]
for t0, t1, txt in transcript_segments:
    print(f"[{t0:5.1f}-{t1:5.1f}s] {txt[:60]}...")


[  0.0-  6.0s] Welcome to the ACME Robotics fiscal year 2024 review. We'll ...
[  6.0- 16.0s] Revenue reached a record twenty-seven million dollars in the...
[ 16.0- 26.0s] Monthly churn spiked to six point two percent in March becau...
[ 26.0- 37.0s] On reliability, APAC ninety-fifth percentile latency reached...
[ 37.0- 45.0s] Finally, the team grew to a hundred and five people, with en...


**Cell 9.3 — Sample keyframes from the video with OpenCV.**
We grab one frame every few seconds (interval sampling — simple and good enough for slides). Each keyframe
is saved with its timestamp so we can line it up with the transcript next. If OpenCV can't open the file,
we fall back to the reference keyframes shipped in the dataset.


In [21]:
def sample_keyframes(path, every_sec=5.0, out_dir=KF):
    import cv2
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 8.0
    frames, i = [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if i % max(1, int(fps*every_sec)) == 0:                 # keep one frame every `every_sec`
            t = i/fps; fp = f"{out_dir}/kf_{int(t):04d}.jpg"
            cv2.imwrite(fp, frame); frames.append((t, fp))
        i += 1
    cap.release(); return frames

try:
    keyframes = sample_keyframes(MP4, every_sec=5.0)
except Exception:
    import glob
    keyframes = [(float(os.path.basename(p).split("_")[1].split(".")[0]), p)
                 for p in sorted(glob.glob(f"{KF}/frame_*.jpg"))]
print(f"{len(keyframes)} keyframes sampled")


9 keyframes sampled


**Cell 9.4 — Align frames to transcript into time-windowed chunks.**
For each transcript segment we attach the keyframe(s) whose timestamp falls inside that window (or the
nearest one). Now every chunk knows *what was said* and *what was on screen* between `t_start` and `t_end`.


In [22]:
def build_video_chunks(segments, frames):
    out = []
    for i,(t0, t1, text) in enumerate(segments):
        imgs = [fp for (t, fp) in frames if t0 <= t < t1] or \
               [min(frames, key=lambda tf: abs(tf[0]-(t0+t1)/2))[1]]   # nearest-frame fallback
        out.append({"id": f"seg_{i:03d}", "t_start": t0, "t_end": t1, "transcript": text, "keyframes": imgs})
    return out

video_chunks = build_video_chunks(transcript_segments, keyframes)
for vc in video_chunks:
    print(f"{vc['id']} [{vc['t_start']:>5.1f}-{vc['t_end']:>5.1f}s] imgs={len(vc['keyframes'])} :: {vc['transcript'][:44]}...")


seg_000 [  0.0-  6.0s] imgs=2 :: Welcome to the ACME Robotics fiscal year 202...
seg_001 [  6.0- 16.0s] imgs=2 :: Revenue reached a record twenty-seven millio...
seg_002 [ 16.0- 26.0s] imgs=2 :: Monthly churn spiked to six point two percen...
seg_003 [ 26.0- 37.0s] imgs=2 :: On reliability, APAC ninety-fifth percentile...
seg_004 [ 37.0- 45.0s] imgs=1 :: Finally, the team grew to a hundred and five...


**Cell 9.5 — Index the video (reusing Strategy C).**
Transcript text goes into a text Qdrant collection (with timestamps in the metadata); keyframes go into
a CLIP image collection. A query searches both and we fuse with the same RRF helper from Section 5.


In [23]:
vid_text_docs = [Document(page_content=vc["transcript"],
                          metadata={"id": vc["id"], "modality": "video_text",
                                    "t_start": vc["t_start"], "t_end": vc["t_end"],
                                    "keyframes": ",".join(vc["keyframes"])}) for vc in video_chunks]
vid_text_store = QdrantVectorStore.from_documents(vid_text_docs, text_embedder,
                                                  location=":memory:", collection_name="video_transcript")

kf_paths = sorted({p for vc in video_chunks for p in vc["keyframes"]})
kf_vecs  = clip.embed_images(kf_paths)
kf_owner = {p: next(vc for vc in video_chunks if p in vc["keyframes"]) for p in kf_paths}
kf_texts = [f"keyframe of {kf_owner[p]['id']}" for p in kf_paths]
kf_metas = [{"id": kf_owner[p]["id"], "modality": "video_frame",
             "t_start": kf_owner[p]["t_start"], "t_end": kf_owner[p]["t_end"], "source_path": p} for p in kf_paths]
vid_img_store = qdrant_store_from_vectors("video_keyframes", kf_texts, kf_vecs, kf_metas,
                                          ClipTextEmbeddings(clip))

def retrieve_video(query, k=3):
    return reciprocal_rank_fusion([vid_text_store.similarity_search(query, k=k),
                                   vid_img_store.similarity_search(query, k=k)])[:k]

for d in retrieve_video("when did the latency alert happen in the video?"):
    m = d.metadata
    print(f"[{m['modality']:11s}] {m['id']} @ {m['t_start']:.0f}-{m['t_end']:.0f}s")


[video_frame] seg_003 @ 26-37s
[video_text ] seg_003 @ 26-37s
[video_text ] seg_000 @ 0-6s


**Cell 9.6 — Answer with a timestamp citation.**
Same pattern as Section 7, but each piece of context is tagged with its `[MM:SS]` window, and keyframes
are sent as real pixels. The model can now point you to the exact moment in the video.


In [24]:
def fmt_ts(sec): return f"{int(sec//60):02d}:{int(sec%60):02d}"

def answer_video(query, k=3):
    docs = retrieve_video(query, k=k)
    parts = [{"type": "text", "text": f"Question: {query}\nAnswer using the video context; cite timestamps."}]
    for d in docs:
        m = d.metadata; stamp = f"[{fmt_ts(m['t_start'])}-{fmt_ts(m['t_end'])}]"
        if m["modality"] == "video_frame":
            parts += [{"type": "text", "text": f"{stamp} keyframe:"}, image_block(m["source_path"])]
        else:
            parts.append({"type": "text", "text": f"{stamp} transcript: {d.page_content}"})
    resp = vlm.invoke([SystemMessage(ANSWER_SYSTEM), HumanMessage(content=parts)])
    return resp.content if isinstance(resp.content, str) else \
           "".join(b.get("text", "") for b in resp.content if isinstance(b, dict))

print(answer_video("When did the APAC latency issue come up and what fixed it?"))


The APAC latency issue came up when the 95th percentile latency reached 240 milliseconds, which was the worst of any region and triggered on-call pages. It was fixed by implementing a regional cache, bringing it back to target [00:26-00:37].


### Scaling video
- **Frame explosion is the enemy.** 30 min @ 30fps ≈ 54k frames — never index them all. Use scene-change
  detection or interval sampling, and dedupe near-identical frames (perceptual hash) before embedding.
- **Chunk on speech, attach vision.** Segment by transcript (sentence/topic windows), attach the
  representative keyframe(s) — coherent and citation-friendly.
- **Parallelize ingestion.** Frame extraction and captioning are embarrassingly parallel — fan them out
  across worker processes or a batch queue rather than looping one video at a time.
- **Long-video retrieval.** Add a coarse "chapter" summary layer (VLM-summarize each few-minute window)
  for hierarchical retrieval.
- **Sync matters.** Carry `t_start/t_end` on every chunk so answers can deep-link (`video.mp4?t=192`).


---
## 🧗 Breakout Room #2 — Check your understanding

Everything you need is in the sections you just ran — if you're stuck, re-read Section 7's introduction,
the callout under Strategy B in Section 5, and Strategy C's description.

### ❓ Question #3:

At generation time we re-load every retrieved image from `source_path` and send the **actual pixels** to
the VLM — even though we already generated a caption for each image during ingestion. Why not just send
the caption?

#### ✅ Answer:

The caption is a lossy ingestion artifact optimized to make the image findable. It may omit a small label,
exact number, legend relationship, or table cell, and any parsing mistake would be repeated if generation
saw only that caption. Reloading the original image preserves the authoritative evidence and lets the VLM
perform detailed chart reading/OCR at query time. Keeping `source_path` therefore separates two jobs cleanly:
the caption or embedding retrieves the likely figure, while the pixels ground the final answer and citation.


### ❓ Question #4:

Strategy C searches a text collection and an image collection, then merges the two result lists with
**Reciprocal Rank Fusion** rather than simply keeping the items with the highest raw similarity scores
across both. Why are raw scores the wrong tool for this merge?

#### ✅ Answer:

Raw similarity scores are not calibrated across the two collections. The text store uses an OpenAI text
embedding model, while the image store uses CLIP; their score distributions, dimensional spaces, and notions
of similarity differ. A `0.80` from one model is therefore not necessarily better evidence than a `0.75`
from the other, so sorting the combined raw scores can systematically favor one modality. RRF uses only each
retriever's rank order and rewards items that rank highly (or appear in multiple lists), producing a simple,
scale-independent merge.


### 🏗️ Activity #2: Extend the evaluation set

Add **two new gold questions** to the Section 8 eval — one whose answer lives in a **text** doc and one
whose answer lives in a **chart image** (use the data dictionary from Section 3 to pick targets). Run
the cell, then record below whether any strategy's recall@3 moved, and why that might be.


In [25]:
# 🏗️ Activity #2 — one text-answerable case and one image-only exact-value case.
my_gold = {
    "Which market segment and geography did ACME prioritize in FY2024?": {"doc_gtm_strategy.md"},
    "What was ACME's lowest quarterly revenue in 2024?": {"fig_revenue_quarterly.png"},
}

strategies = ["A", "B", "C"]
# Keep the cell idempotent: remove these cases before recomputing the 8-question baseline.
for q in my_gold:
    gold.pop(q, None)
baseline_scores = {s: recall_at_k(s) for s in strategies}
gold.update(my_gold)
extended_scores = {s: recall_at_k(s) for s in strategies}

for s in strategies:
    delta = extended_scores[s] - baseline_scores[s]
    print(f"Strategy {s}: recall@3 = {extended_scores[s]:.2f} "
          f"(change {delta:+.2f}, n={len(gold)} questions)")

print("\nNew-case diagnostics:")
for q, want in my_gold.items():
    print(f"Q: {q}")
    for s in strategies:
        got = {d["id"] for d in MultiModalRetriever(s, 3).retrieve(q)}
        print(f"  Strategy {s}: hit={bool(want & got)} retrieved={sorted(got)}")


Strategy A: recall@3 = 1.00 (change +0.00, n=10 questions)
Strategy B: recall@3 = 0.30 (change +0.05, n=10 questions)
Strategy C: recall@3 = 1.00 (change +0.00, n=10 questions)

New-case diagnostics:
Q: Which market segment and geography did ACME prioritize in FY2024?


  Strategy A: hit=True retrieved=['doc_board_notes.md', 'doc_gtm_strategy.md', 'fig_revenue_quarterly.png']
  Strategy B: hit=True retrieved=['doc_board_notes.md', 'doc_gtm_strategy.md', 'doc_people_ops.md']


  Strategy C: hit=True retrieved=['doc_board_notes.md', 'doc_gtm_strategy.md', 'fig_headcount_dept.png']
Q: What was ACME's lowest quarterly revenue in 2024?


  Strategy A: hit=True retrieved=['doc_board_notes.md', 'doc_gtm_strategy.md', 'fig_revenue_quarterly.png']
  Strategy B: hit=False retrieved=['doc_board_notes.md', 'doc_gtm_strategy.md', 'doc_people_ops.md']


  Strategy C: hit=True retrieved=['doc_board_notes.md', 'doc_gtm_strategy.md', 'fig_revenue_quarterly.png']


**Your observations:**

The extended evaluation contains ten questions and adds complementary failure modes. The text question was
retrieved successfully by all three strategies through `doc_gtm_strategy.md`. The exact-value image question
was retrieved by Strategies A and C through `fig_revenue_quarterly.png`, but Strategy B again returned only text
documents and missed the figure because of the unified-store modality gap.

The baseline recall@3 scores were **A = 1.00, B = 0.25, C = 1.00**. After adding the two cases they were
**A = 1.00, B = 0.30, C = 1.00**. Strategy B improved by `+0.05` because it hit the new text case, but its
image weakness remained visible. A and C stayed perfect on this small corpus. The per-question diagnostics
make clear that the score movement came from a concrete text hit rather than improved image retrieval.

---
#### 🛑 End of Breakout Room #2 — you built the whole pipeline. Recap below.


<a id="sec10"></a>
## 10. Recap: what we built vs. production

**You built:** a VLM used two ways (ingestion parser + query-time reader); **three retrieval strategies**
with a tradeoff table and a one-line-swappable retriever; a **Qdrant** pipeline where a text question
retrieves a chart image and the model reads the exact number off it; and a **video + transcript**
extension with timestamp citations — all against a **CC0 synthetic dataset** that ships with the repo.

### What we built today vs. what production looks like

| Component | What we built today | What production looks like |
|---|---|---|
| Vector store | Qdrant in `:memory:` mode, rebuilt on every run | Managed/self-hosted Qdrant cluster: persistent collections, replication, snapshots, incremental upserts |
| Corpus | 6 charts + 6 docs + one 45s video, all synthetic | Millions of PDF pages, slides, and screenshots, rasterized page-by-page with provenance on every chunk |
| Image parsing | A sequential loop of 6 VLM calls, no error handling | Parallel ingestion workers with retries, JSON-schema validation, dead-letter queues, and caching |
| Embeddings | CLIP ViT-B/32 running locally in the notebook | A served embedding endpoint (or batch jobs), version-pinned so the index and queries never drift |
| Retrieval | Top-3 cosine similarity + RRF | Hybrid (BM25 + dense) search, metadata filters, cross-encoder reranking, per-modality tuning |
| Generation | One prompt with every retrieved image attached | Token budgets, image downscaling, prompt caching, citation post-checks, guardrails, fallbacks |
| Evaluation | recall@3 over 8 hand-written gold questions | Continuous eval suites (retrieval + answer quality), LLM-as-judge, regression gates in CI |
| Video | Interval keyframe sampling on one short clip | Scene-change detection, perceptual-hash dedupe, hierarchical chapter summaries, deep links |
| Secrets & config | A local `.env` file | A secrets manager, per-environment config, key rotation, cost monitoring, rate-limit handling |
| Serving | Cells in a notebook | An API service with tracing/monitoring (e.g. LangSmith), autoscaling, and access control |

The gap between those two columns is the rest of your journey as an AI Engineer — the *concepts* you
just built are the same ones running inside production systems; production adds scale, reliability, and
observability around them.


### 🌱 Continued Learning (post-cohort stretch goals)

Focus on your certification requirements during the cohort. Come back to these afterward to reinforce
and extend what you built today:

1. Change `VLM_MODEL`/`EMBED_MODEL` to another provider and re-run — does the `recall@3` table change?
2. Add **hybrid retrieval** (BM25 keyword + dense) on the text side and fuse three lists with RRF.
3. Enrich the parse prompt to extract **Markdown tables** from figures; see if Strategy A improves on
   numeric questions.
4. Add a **cross-encoder reranker** after fusion and check whether top-1 accuracy improves.
5. For video, replace interval sampling with **scene-change detection** + perceptual-hash dedupe.
6. Ask something the data can't answer ("what was 2023 revenue?") and confirm the model declines instead
   of hallucinating.

**Further reading (verify current versions):** LangChain multimodal inputs & content blocks,
`init_chat_model`; CLIP / `sentence-transformers`; Qdrant; Reciprocal Rank Fusion.
